# Benchmark v2 review

100 questions on the unchanged v1 snapshot. Start here for the question set and recorded checks.

This is an evaluator notebook: do not give it or the answer key to the model. Human review is pending. All cells are offline.


## How to use the next cell
Run from the repository root or a descendant with Python 3.10+. Press Shift+Enter to load the generated inventory and show split/category counts. No packages or credentials are required.


In [ ]:
from pathlib import Path
import json
from collections import Counter
repo = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'fa26/benchmark/v2/README.md').exists())
root = repo/'fa26/benchmark/v2'
questions = json.loads((root/'evaluator_catalog.json').read_text())
reference = json.loads((root/'reference_answers.json').read_text())
print('Questions:', len(questions))
print('Splits:', dict(Counter(q['split'] for q in questions)))
print('New categories:', dict(Counter(q['category'] for q in questions if not q['legacy'])))


## How to use the next cell: ten review examples
Run the first cell, then this cell. Check the question, calculation scope and expected behaviour against the source data. Full row-level evidence is in reference_answers.json. Mark human approval outside the generated files until the team agrees on the final contracts.


In [ ]:
example_ids = set(json.loads((root/'review_examples.json').read_text())['question_ids'])
answers = {a['id']: a for a in reference['answers']}
for q in questions:
    if q['id'] in example_ids:
        a = answers[q['id']]
        print(q['id'], q['question'])
        print(q['contract'])
        print('Expected:', a['status'], a['reasons'])
        print('Facts:', {k: {'value': v['value'], 'evidence_count': len(v['evidence'])} for k,v in a['facts'].items()})
        print(q.get('explanation', ''), '\n')


## How to use the next cell: recorded verification
Displays the synthetic SQL baseline and split scores. These are offline reference checks, not evidence of real model quality.


In [ ]:
baseline = json.loads((root/'results/offline_baseline.json').read_text())
print(baseline['verification'])
print('Live model runs:', baseline['live_model_runs'])
print(baseline['result']['summary'])
print(baseline['result']['by_split'])


## How to use the next cell: optional checks
Leave RUN_CHECKS=False for reading only. Set it to True to recalculate and test the reference in memory. Tests use temporary exports; they do not call APIs or overwrite the saved answers. Run build.py separately only when intentionally regenerating evaluator artifacts.


In [ ]:
RUN_CHECKS = False
if RUN_CHECKS:
    import subprocess, sys
    subprocess.run([sys.executable, str(root/'test_v2.py')], cwd=repo, check=True)
else:
    print('Checks not rerun. See README.md for offline commands.')


## Handoff

1. Review ten examples, then the full new question set with John.
2. Export only development inputs while tuning.
3. Adapt each workflow to the v2 IDs and submission metadata.
4. Confirm access/budget before live runs.
5. Freeze first held-out results before changing templates based on failures.

See [README](README.md) for commands and [answer review](ANSWER_KEY.md) for contracts. The reference checks share specifications; human wording review is still needed.
